# SAMQ — Sensitivity-Aware Mixed-Precision Quantization-Aware Fine-Tuning below 4 bits

Companion notebook to the paper *"Where Do Low-Bit Language Models Break? Sensitivity-Aware Mixed-Precision
Quantization-Aware Fine-Tuning for 2–3-bit Transformer Language Models"*.

The notebook reproduces every experiment in the paper end to end:

| § | What it does |
|---|---|
| 1 | Setup, configuration |
| 2 | Library: group-wise quantiser, LSQ, MSE clipping, GPTQ, Fisher, ILP allocation, QAT trainer |
| 3 | WikiText-2 + byte-level BPE (8 192 tokens) |
| 4 | Pre-train the FP32 reference decoder (6 layers, d=256, 6.9 M parameters) |
| 5 | **Which layers break first?** single-layer bit sweeps, type/depth groups, embedding, proxies, additivity |
| 6 | Post-training baselines (RTN, RTN+MSE clip, GPTQ) and group-size sweep |
| 7 | Bit-allocation policies (ILP on measured sensitivity / Fisher / Hessian / weight-MSE, heuristics, random) |
| 8 | Quantization-aware fine-tuning: STE, LSQ, LSQ+KD, **SAMQ** (uniform and mixed precision), ablations, seeds |
| 9 | Result tables, Pareto frontier, error propagation, oscillation analysis |
| 10 | Real bit-packed export (exact storage) and edge-device projections |
| 11 | *Optional:* the same pipeline on a pretrained Hugging Face LLM (SmolLM2 / Qwen2.5) |

**Runtime.** On a Colab T4 GPU the full paper configuration takes roughly 1–2 h; set `FAST = True` for a ~10 min smoke run.
Everything also runs on CPU (that is how the paper's numbers were produced), just slower.

In [ ]:
!pip -q install tokenizers scipy pandas matplotlib
import os, math, json, time, copy, random, urllib.request
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, kendalltau

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| torch', torch.__version__)

FAST = False          # True -> tiny smoke-test version of every experiment

CFG = dict(
    # reference model (paper: d=256, L=6, h=4, ff=1024, T=256)
    d=256, L=6, h=4, ff=1024, T=256, vocab=8192,
    pretrain_steps=3000, pretrain_bs=16, pretrain_lr=1e-3,
    # quantisation
    group=64,
    # QAT (paper values; lr / hidden-distillation weight were picked on the validation set)
    qat_steps=250, qat_bs=16, qat_lr=5e-4, qat_qlr=1e-3,
    sens_windows=128,     # validation windows (x256 tokens) used for sensitivity probes
    test_windows=None,    # None = full WikiText-2 test split
)
if FAST:
    CFG.update(pretrain_steps=300, qat_steps=40, sens_windows=16, test_windows=64)
if os.environ.get('NB_SMOKE'):   # used only to smoke-test this notebook on CPU
    CFG.update(d=64, L=2, ff=256, pretrain_steps=60, qat_steps=6, sens_windows=4, test_windows=8, smoke=True)
os.makedirs('runs', exist_ok=True)

## 2. Library
Identical to `qlib.py` used for the paper, plus device handling and real bit-packing.

In [ ]:
"""
qlib.py -- Sub-4-bit quantization-aware fine-tuning toolkit.

Contents
  * WikiText-2 byte-level BPE data pipeline
  * Pre-LN GPT decoder whose every linear layer is a QLinear (fake-quant capable)
  * Group-wise asymmetric uniform weight quantizer with three modes:
        - 'minmax' : scales recomputed from current weights (STE through round)
        - 'lsq'    : learnable per-group step size and offset (LSQ/LSQ+ style)
        - frozen   : fixed scales (used for PTQ / GPTQ export)
  * MSE-optimal clipping search for scale initialisation
  * GPTQ (Frantar et al., 2023) with group-wise parameters
  * Sensitivity measurement (empirical delta-NLL, Fisher, Hessian-weighted error, kurtosis)
  * Exact ILP bit allocation (scipy.optimize.milp)
  * QAT loop with optional logit distillation, hidden-state distillation and
    progressive bit annealing
"""
import math, os, json, time, copy, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ----------------------------------------------------------------------------
# Reproducibility
# ----------------------------------------------------------------------------
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

# ----------------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------------
def build_data(data_dir, vocab_size=8192, cache='wt2_bpe.pt'):
    path = os.path.join(data_dir, cache)
    if os.path.exists(path):
        return torch.load(path)
    from tokenizers import ByteLevelBPETokenizer
    files = {s: os.path.join(data_dir, f'wt2_{s}.txt') for s in ['train', 'valid', 'test']}
    tok = ByteLevelBPETokenizer()
    tok.train([files['train']], vocab_size=vocab_size, min_frequency=2,
              special_tokens=['<|endoftext|>'])
    out = {}
    for s, f in files.items():
        with open(f, encoding='utf-8') as fh:
            text = fh.read()
        out[s] = torch.tensor(tok.encode(text).ids, dtype=torch.int32)
    out['vocab_size'] = tok.get_vocab_size()
    tok.save_model(data_dir, 'wt2bpe')
    torch.save(out, path)
    return out

def get_batch(data, bs, T, gen=None, device='cpu'):
    ix = torch.randint(0, len(data) - T - 1, (bs,), generator=gen)
    x = torch.stack([data[i:i + T] for i in ix]).long()
    y = torch.stack([data[i + 1:i + T + 1] for i in ix]).long()
    return x.to(device), y.to(device)

def dev(model):
    return next(model.parameters()).device

# ----------------------------------------------------------------------------
# Quantizer
# ----------------------------------------------------------------------------
def round_ste(x):
    return (x.round() - x).detach() + x

def _grouped(w, g):
    """(out, in) -> (out, in/g, g)"""
    o, i = w.shape
    assert i % g == 0, (i, g)
    return w.reshape(o, i // g, g)

@torch.no_grad()
def minmax_params(w, bits, g, alpha=None):
    """Asymmetric per-group scale/zero. alpha: (out, ngroups) shrink ratio for clipping."""
    wg = _grouped(w, g)
    mx = wg.amax(-1); mn = wg.amin(-1)
    if alpha is not None:
        mx = mx * alpha; mn = mn * alpha
    qmax = 2 ** bits - 1
    s = ((mx - mn) / qmax).clamp(min=1e-8)
    z = (-mn / s).round().clamp(0, qmax)
    return s, z

def fake_quant(w, s, z, bits, g, ste=True):
    """Group-wise asymmetric uniform fake quantisation. s, z: (out, ngroups)."""
    qmax = 2 ** bits - 1
    wg = _grouped(w, g)
    s_ = s.unsqueeze(-1); z_ = z.unsqueeze(-1)
    r = round_ste if ste else torch.round
    q = torch.clamp(r(wg / s_) + z_, 0, qmax)
    return ((q - z_) * s_).reshape(w.shape)

@torch.no_grad()
def mse_clip_search(w, bits, g, grid=20, lo=0.5, p=2.0):
    """Per-group grid search of clipping ratio alpha minimising ||w - Q(w)||_p^p."""
    wg = _grouped(w, g)
    best = torch.full(wg.shape[:2], float('inf'), device=w.device)
    best_a = torch.ones(wg.shape[:2], device=w.device)
    for a in torch.linspace(lo, 1.0, grid):
        A = torch.full(wg.shape[:2], float(a), device=w.device)
        s, z = minmax_params(w, bits, g, A)
        err = (fake_quant(w, s, z, bits, g, ste=False) - w).abs().pow(p)
        err = _grouped(err, g).sum(-1)
        m = err < best
        best[m] = err[m]; best_a[m] = float(a)
    return best_a

class QLinear(nn.Linear):
    """nn.Linear with optional group-wise fake quantisation of the weight."""
    def __init__(self, i, o, bias=True):
        super().__init__(i, o, bias)
        self.qmode = None      # None | 'minmax' | 'lsq' | 'frozen'
        self.bits = 16
        self.g = 64
        self.name = ''

    def configure(self, bits, mode, g=64, clip='mse'):
        if g == 0: g = self.weight.shape[1]          # g=0 -> per-output-channel
        self.bits, self.qmode, self.g, self.clip = bits, mode, g, clip
        if mode is None or bits >= 16:
            self.qmode = None; return
        w = self.weight.data
        alpha = mse_clip_search(w, bits, g) if clip == 'mse' else None
        s, z = minmax_params(w, bits, g, alpha)
        if mode == 'lsq':
            # learn log-step and continuous zero point (rounded with STE in fwd)
            self.log_s = nn.Parameter(s.log().clone())
            self.zp = nn.Parameter(z.clone())
        else:
            self.register_buffer('s_buf', s.clone(), persistent=False)
            self.register_buffer('z_buf', z.clone(), persistent=False)
        self.alpha = alpha

    def rebits(self, bits, clip=None):
        """Change bit-width in place (progressive annealing), re-initialising scales."""
        mode = self.qmode
        for n in ['log_s', 'zp']:
            if hasattr(self, n): delattr(self, n)
        for n in ['s_buf', 'z_buf']:
            if n in self._buffers: del self._buffers[n]
        self.configure(bits, mode, self.g, clip or self.clip)

    def qweight(self):
        if self.qmode is None:
            return self.weight
        if self.qmode == 'minmax':
            s, z = minmax_params(self.weight, self.bits, self.g, self.alpha)
            return fake_quant(self.weight, s, z, self.bits, self.g)
        if self.qmode == 'lsq':
            qmax = 2 ** self.bits - 1
            s = self.log_s.exp()
            z = torch.clamp(round_ste(self.zp), 0, qmax)
            return fake_quant(self.weight, s, z, self.bits, self.g)
        if self.qmode == 'frozen':
            return fake_quant(self.weight, self.s_buf, self.z_buf, self.bits, self.g, ste=False)

    def forward(self, x):
        return F.linear(x, self.qweight(), self.bias)

# ----------------------------------------------------------------------------
# Model
# ----------------------------------------------------------------------------
class Block(nn.Module):
    def __init__(self, d, h, ff, drop):
        super().__init__()
        self.h = h
        self.ln1 = nn.LayerNorm(d); self.ln2 = nn.LayerNorm(d)
        self.q = QLinear(d, d); self.k = QLinear(d, d); self.v = QLinear(d, d)
        self.o = QLinear(d, d)
        self.fc1 = QLinear(d, ff); self.fc2 = QLinear(ff, d)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        B, T, C = x.shape
        a = self.ln1(x)
        q = self.q(a).view(B, T, self.h, -1).transpose(1, 2)
        k = self.k(a).view(B, T, self.h, -1).transpose(1, 2)
        v = self.v(a).view(B, T, self.h, -1).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=self.drop.p if self.training else 0.0)
        y = y.transpose(1, 2).reshape(B, T, C)
        x = x + self.drop(self.o(y))
        x = x + self.drop(self.fc2(F.gelu(self.fc1(self.ln2(x)))))
        return x

class GPT(nn.Module):
    def __init__(self, vocab, d=256, L=6, h=4, ff=1024, T=256, drop=0.1):
        super().__init__()
        self.cfg = dict(vocab=vocab, d=d, L=L, h=h, ff=ff, T=T, drop=drop)
        self.wte = nn.Embedding(vocab, d)
        self.wpe = nn.Embedding(T, d)
        self.blocks = nn.ModuleList([Block(d, h, ff, drop) for _ in range(L)])
        self.lnf = nn.LayerNorm(d)
        self.drop = nn.Dropout(drop)
        self.apply(self._init)
        for b in self.blocks:   # GPT-2 residual scaling
            nn.init.normal_(b.o.weight, 0, 0.02 / math.sqrt(2 * L))
            nn.init.normal_(b.fc2.weight, 0, 0.02 / math.sqrt(2 * L))
        for li, b in enumerate(self.blocks):
            for n in ['q', 'k', 'v', 'o', 'fc1', 'fc2']:
                getattr(b, n).name = f'{li}.{n}'

    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0, 0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0, 0.02)

    def forward(self, idx, return_hidden=False):
        B, T = idx.shape
        x = self.drop(self.wte(idx) + self.wpe(torch.arange(T, device=idx.device)))
        hs = []
        for b in self.blocks:
            x = b(x)
            if return_hidden: hs.append(x)
        x = self.lnf(x)
        logits = x @ self.wte.weight.t()          # tied LM head
        return (logits, hs) if return_hidden else logits

    def qlinears(self):
        return [m for m in self.modules() if isinstance(m, QLinear)]

def nparams(model, linear_only=False):
    if linear_only:
        return sum(m.weight.numel() for m in model.qlinears())
    return sum(p.numel() for p in model.parameters())

# ----------------------------------------------------------------------------
# Evaluation
# ----------------------------------------------------------------------------
@torch.no_grad()
def evaluate(model, data, T=256, max_windows=None, bs=16):
    """Perplexity with non-overlapping windows; also top-1 next-token accuracy."""
    model.eval()
    n = (len(data) - 1) // T
    if max_windows: n = min(n, max_windows)
    nll, cnt, correct = 0.0, 0, 0
    for i in range(0, n, bs):
        idx = torch.arange(i, min(i + bs, n)) * T
        x = torch.stack([data[j:j + T] for j in idx]).long().to(dev(model))
        y = torch.stack([data[j + 1:j + T + 1] for j in idx]).long().to(dev(model))
        lg = model(x)
        nll += F.cross_entropy(lg.reshape(-1, lg.size(-1)), y.reshape(-1), reduction='sum').item()
        correct += (lg.argmax(-1) == y).sum().item()
        cnt += y.numel()
    return dict(ppl=math.exp(nll / cnt), nll=nll / cnt, acc=correct / cnt)

# ----------------------------------------------------------------------------
# Quantisation configuration helpers
# ----------------------------------------------------------------------------
def quantize_model(model, bits_map, mode, g=64, clip='mse'):
    """bits_map: dict layer-name -> bits, or an int for uniform."""
    for m in model.qlinears():
        b = bits_map if isinstance(bits_map, int) else bits_map.get(m.name, 16)
        m.configure(b, mode, g, clip)
    return model

def bake(model):
    """Replace weights by their quantised values and drop the quantiser (export)."""
    with torch.no_grad():
        for m in model.qlinears():
            if m.qmode is not None:
                m.weight.data = m.qweight().detach().clone()
                m.qmode = None
    return model

def bits_report(model, g=64, scale_bits=16):
    """Average weight bits over quantised linears incl. per-group scale+zero overhead."""
    tot, bits_sum, eff = 0, 0, 0
    for m in model.qlinears():
        n = m.weight.numel(); tot += n
        b = m.bits if m.qmode is not None else 16
        bits_sum += b * n
        over = (scale_bits + b) / m.g if b < 16 else 0   # fp16 scale + b-bit zero per group
        eff += (b + over) * n
    return dict(avg_bits=bits_sum / tot, eff_bits=eff / tot)

# ----------------------------------------------------------------------------
# GPTQ
# ----------------------------------------------------------------------------
@torch.no_grad()
def collect_inputs(model, calib, layer_names=None):
    """Return H = 2 X^T X / N for every QLinear's input on calibration batches."""
    Hs, cnt, hooks = {}, {}, []
    for m in model.qlinears():
        if layer_names and m.name not in layer_names: continue
        def hook(mod, inp, out):
            x = inp[0].reshape(-1, inp[0].shape[-1]).double()
            Hs[mod.name] = Hs.get(mod.name, 0) + 2 * x.t() @ x
            cnt[mod.name] = cnt.get(mod.name, 0) + x.shape[0]
        hooks.append(m.register_forward_hook(hook))
    model.eval()
    for x in calib: model(x)
    for h in hooks: h.remove()
    return {k: (v / cnt[k]).float() for k, v in Hs.items()}

@torch.no_grad()
def gptq_layer(W, H, bits, g, blocksize=128, percdamp=0.01, clip='mse', actorder=False):
    W = W.clone().float(); ncol = W.shape[1]
    H = H.clone()
    dead = torch.diag(H) == 0
    H[dead, dead] = 1; W[:, dead] = 0
    perm = None
    if actorder:
        perm = torch.argsort(torch.diag(H), descending=True)
        W = W[:, perm]; H = H[perm][:, perm]
    damp = percdamp * torch.mean(torch.diag(H))
    H += damp * torch.eye(ncol, device=H.device)
    Hinv = torch.linalg.cholesky(torch.cholesky_inverse(torch.linalg.cholesky(H)), upper=True)
    Q = torch.zeros_like(W)
    qmax = 2 ** bits - 1
    s = z = None
    for i1 in range(0, ncol, blocksize):
        i2 = min(i1 + blocksize, ncol)
        W1 = W[:, i1:i2].clone(); Q1 = torch.zeros_like(W1); Err1 = torch.zeros_like(W1)
        Hinv1 = Hinv[i1:i2, i1:i2]
        for i in range(i2 - i1):
            col = i1 + i
            if col % g == 0:
                blk = W[:, col:col + g].clone()
                blk[:, :min(g, i2 - col)] = W1[:, i:i + min(g, i2 - col)]
                if col + g > i2:   # group straddles block: use updated tail where known
                    blk[:, i2 - col:] = W[:, i2:col + g]
                alpha = mse_clip_search(blk, bits, g).squeeze(1) if clip == 'mse' else None
                mx = blk.amax(1); mn = blk.amin(1)
                if alpha is not None: mx, mn = mx * alpha, mn * alpha
                s = ((mx - mn) / qmax).clamp(min=1e-8); z = (-mn / s).round().clamp(0, qmax)
            w = W1[:, i]; d = Hinv1[i, i]
            q = (torch.clamp(torch.round(w / s) + z, 0, qmax) - z) * s
            Q1[:, i] = q
            err = (w - q) / d
            W1[:, i:] -= err.unsqueeze(1) @ Hinv1[i, i:].unsqueeze(0)
            Err1[:, i] = err
        Q[:, i1:i2] = Q1
        W[:, i2:] -= Err1 @ Hinv[i1:i2, i2:]
    if perm is not None:
        Q = Q[:, torch.argsort(perm)]
    return Q

@torch.no_grad()
def gptq_quantize(model, calib, bits_map, g=64, clip='mse'):
    """Sequential, block-by-block GPTQ with 'true-sequential' sub-groups: inputs of a
    sub-layer are produced by the already-quantised preceding sub-layers and blocks."""
    model.eval()
    xs = []
    for x in calib:
        T = x.shape[1]
        xs.append(model.wte(x) + model.wpe(torch.arange(T, device=x.device)))
    for li, blk in enumerate(model.blocks):
        for grp in [['q', 'k', 'v'], ['o'], ['fc1'], ['fc2']]:
            Hs, cnt, hooks = {}, {}, []
            for n in grp:
                mod = getattr(blk, n)
                def hook(mod, inp, out):
                    a = inp[0].reshape(-1, inp[0].shape[-1]).double()
                    Hs[mod.name] = Hs.get(mod.name, 0) + 2 * a.t() @ a
                    cnt[mod.name] = cnt.get(mod.name, 0) + a.shape[0]
                hooks.append(mod.register_forward_hook(hook))
            for h in xs: blk(h)
            for h in hooks: h.remove()
            for n in grp:
                m = getattr(blk, n)
                b = bits_map if isinstance(bits_map, int) else bits_map.get(m.name, 16)
                if b >= 16: continue
                gg = g if g else m.weight.shape[1]
                m.weight.data = gptq_layer(m.weight.data, (Hs[m.name] / cnt[m.name]).float(), b, gg, clip=clip)
        xs = [blk(h) for h in xs]
    return model

# ----------------------------------------------------------------------------
# Sensitivity
# ----------------------------------------------------------------------------
def kurtosis(w):
    w = w.flatten().double(); m = w.mean(); v = ((w - m) ** 2).mean()
    return (((w - m) ** 4).mean() / v ** 2).item()

def empirical_fisher(model, data, nb=16, bs=8, T=256, seed=0):
    """Diagonal empirical Fisher (mean squared gradient) for every QLinear weight."""
    g = torch.Generator().manual_seed(seed)
    model.eval()
    fis = {m.name: torch.zeros_like(m.weight) for m in model.qlinears()}
    for _ in range(nb):
        x, y = get_batch(data, bs, T, g, dev(model))
        model.zero_grad()
        lg = model(x)
        F.cross_entropy(lg.reshape(-1, lg.size(-1)), y.reshape(-1)).backward()
        for m in model.qlinears():
            fis[m.name] += m.weight.grad.detach() ** 2 / nb
    model.zero_grad()
    return fis

# ----------------------------------------------------------------------------
# ILP bit allocation
# ----------------------------------------------------------------------------
def allocate_bits(cost, sizes, choices, budget_avg, fixed=None):
    """min sum_l cost[l][b] x_lb  s.t. sum_b x_lb = 1, sum_lb size_l b x_lb <= budget_avg*sum size.
    cost: dict name -> dict bits -> value. Solved exactly with scipy MILP (HiGHS)."""
    from scipy.optimize import milp, LinearConstraint, Bounds
    names = list(cost.keys()); L = len(names); K = len(choices)
    c = np.array([cost[n][b] for n in names for b in choices], dtype=float)
    A_one = np.zeros((L, L * K))
    for i in range(L): A_one[i, i * K:(i + 1) * K] = 1
    tot = sum(sizes[n] for n in names)
    A_bud = np.array([[sizes[n] * b for n in names for b in choices]], dtype=float) / tot
    cons = [LinearConstraint(A_one, 1, 1), LinearConstraint(A_bud, -np.inf, budget_avg + 1e-9)]
    res = milp(c, constraints=cons, integrality=np.ones(L * K), bounds=Bounds(0, 1))
    assert res.success, res.message
    x = res.x.reshape(L, K).round()
    return {n: choices[int(x[i].argmax())] for i, n in enumerate(names)}

# ----------------------------------------------------------------------------
# QAT
# ----------------------------------------------------------------------------
def qat_train(student, train, valid, steps=600, bs=16, T=256, lr=2e-4, qlr=1e-3,
              teacher=None, kd='logit', kd_T=1.0, hid_w=0.0, anneal=None, seed=0,
              log_every=100, eval_windows=64, warmup=30, verbose=True, callback=None):
    """Quantisation-aware fine-tuning.
    kd: None (CE only) | 'logit' (KL to teacher) | 'mix' (0.5 CE + 0.5 KL)
    hid_w: weight on per-block hidden-state MSE distillation
    anneal: None or dict(final=bits_map, start_bits=4, frac=0.5) progressive bit lowering
    """
    set_seed(seed)
    gen = torch.Generator().manual_seed(seed)
    student.train()
    if teacher is not None: teacher.eval()

    def make_opt():
        wp = [p for n, p in student.named_parameters() if not (n.endswith('log_s') or n.endswith('zp'))]
        qp = [p for n, p in student.named_parameters() if (n.endswith('log_s') or n.endswith('zp'))]
        groups = [{'params': wp, 'lr': lr, 'weight_decay': 0.0}]
        if qp: groups.append({'params': qp, 'lr': qlr, 'weight_decay': 0.0})
        return torch.optim.AdamW(groups, betas=(0.9, 0.95))
    opt = make_opt()
    base = [gr['lr'] for gr in opt.param_groups]

    # progressive annealing schedule: each layer moves start_bits -> target in stages
    stages = []
    if anneal:
        final = anneal['final']; sb = anneal.get('start_bits', 4)
        mids = sorted({b for b in range(min(final.values()), sb)}, reverse=True)  # e.g. [3,2]
        n_st = len(mids)
        for k, b in enumerate(mids):
            at = int(anneal.get('frac', 0.5) * steps * (k + 1) / (n_st + 1))
            stages.append((at, b))
    hist = []
    t0 = time.time()
    for step in range(steps):
        for at, b in stages:
            if step == at:
                for m in student.qlinears():
                    tgt = anneal['final'].get(m.name, 16)
                    if m.qmode is not None and m.bits > max(b, tgt):
                        m.rebits(max(b, tgt))
                opt = make_opt()
        frac = step / steps
        mult = min(1.0, (step + 1) / warmup) * 0.5 * (1 + math.cos(math.pi * frac))
        for gr, b0 in zip(opt.param_groups, base): gr['lr'] = b0 * mult
        x, y = get_batch(train, bs, T, gen, dev(student))
        if hid_w > 0:
            lg, hs = student(x, return_hidden=True)
        else:
            lg = student(x)
        ce = F.cross_entropy(lg.reshape(-1, lg.size(-1)), y.reshape(-1))
        loss = ce
        if teacher is not None and kd:
            with torch.no_grad():
                if hid_w > 0: tl, ths = teacher(x, return_hidden=True)
                else: tl = teacher(x)
            kl = F.kl_div(F.log_softmax(lg / kd_T, -1), F.log_softmax(tl / kd_T, -1),
                          log_target=True, reduction='none').sum(-1).mean() * kd_T ** 2
            loss = kl if kd == 'logit' else 0.5 * ce + 0.5 * kl
            if hid_w > 0:
                hl = sum(F.mse_loss(a, b) / b.pow(2).mean() for a, b in zip(hs, ths)) / len(hs)
                loss = loss + hid_w * hl
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        opt.step()
        if callback is not None: callback(step + 1)
        if (step + 1) % log_every == 0 or step == steps - 1:
            ev = evaluate(student, valid, T, max_windows=eval_windows)
            student.train()
            hist.append(dict(step=step + 1, loss=loss.item(), ce=ce.item(), val_ppl=ev['ppl'],
                             t=time.time() - t0))
            if verbose:
                print(f'  step {step+1:5d} loss {loss.item():.4f} ce {ce.item():.4f} '
                      f'val_ppl {ev["ppl"]:.2f} ({time.time()-t0:.0f}s)', flush=True)
    student.eval()
    return hist

In [ ]:
"""packing.py -- real bit-packed export of a quantised model and exact size accounting."""
import numpy as np
import torch

def pack_codes(codes, bits):
    """codes: uint8 array of values in [0, 2^bits). Returns bytes using exactly ceil(N*bits/8) bytes."""
    c = np.asarray(codes, dtype=np.uint8).ravel()
    planes = np.unpackbits(c[:, None], axis=1)[:, 8 - bits:]      # (N, bits), MSB first
    return np.packbits(planes.ravel()).tobytes(), c.size

def unpack_codes(buf, n, bits):
    planes = np.unpackbits(np.frombuffer(buf, dtype=np.uint8))[:n * bits].reshape(n, bits)
    full = np.zeros((n, 8), dtype=np.uint8); full[:, 8 - bits:] = planes
    return np.packbits(full, axis=1).ravel()

@torch.no_grad()
def export_linear(W, s, z, bits, g):
    """Quantise W (out,in) with group params s,z (out, in/g). Returns dict of packed buffers."""
    qmax = 2 ** bits - 1
    s = s.to(torch.float16).to(torch.float32)          # scales are stored in fp16
    o, i = W.shape
    wg = W.reshape(o, i // g, g)
    q = torch.clamp(torch.round(wg / s.unsqueeze(-1)) + z.unsqueeze(-1), 0, qmax).to(torch.uint8)
    codes, n = pack_codes(q.cpu().numpy(), bits)
    zeros, nz = pack_codes(z.to(torch.uint8).cpu().numpy(), bits)
    scales = s.to(torch.float16).cpu().numpy().tobytes()
    return dict(codes=codes, n=n, zeros=zeros, nz=nz, scales=scales, shape=(o, i), bits=bits, g=g)

def dequant_linear(e):
    o, i = e['shape']; g = e['g']; b = e['bits']
    q = unpack_codes(e['codes'], e['n'], b).reshape(o, i // g, g).astype(np.float32)
    z = unpack_codes(e['zeros'], e['nz'], b).reshape(o, i // g, 1).astype(np.float32)
    s = np.frombuffer(e['scales'], dtype=np.float16).reshape(o, i // g, 1).astype(np.float32)
    return torch.from_numpy(((q - z) * s).reshape(o, i))

def nbytes(e):
    return len(e['codes']) + len(e['zeros']) + len(e['scales'])

## 3. Data — WikiText-2 with a byte-level BPE tokenizer trained on the training split

In [ ]:
os.makedirs('data', exist_ok=True)
BASE = 'https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/'
for s in ['train', 'valid', 'test']:
    f = f'data/wt2_{s}.txt'
    if not os.path.exists(f):
        urllib.request.urlretrieve(BASE + f'{s}.txt', f)
D = build_data('data', vocab_size=CFG['vocab'])
print({k: (len(v) if hasattr(v, '__len__') else v) for k, v in D.items()})

## 4. Pre-train the full-precision reference model

In [ ]:
def pretrain(steps, bs=16, T=256, LR=1e-3, WU=200, seed=0):
    set_seed(seed)
    m = GPT(D['vocab_size'], d=CFG['d'], L=CFG['L'], h=CFG['h'], ff=CFG['ff'], T=T, drop=0.1).to(DEVICE)
    decay = [p for n, p in m.named_parameters() if p.dim() >= 2]
    nodecay = [p for n, p in m.named_parameters() if p.dim() < 2]
    opt = torch.optim.AdamW([{'params': decay, 'weight_decay': 0.1}, {'params': nodecay, 'weight_decay': 0.0}],
                            lr=LR, betas=(0.9, 0.95))
    gen = torch.Generator().manual_seed(seed); t0 = time.time()
    for step in range(steps):
        lr = LR * min(1, (step + 1) / WU) * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * step / steps)))
        for g in opt.param_groups: g['lr'] = lr
        m.train(); x, y = get_batch(D['train'], bs, T, gen, DEVICE)
        loss = F.cross_entropy(m(x).flatten(0, 1), y.flatten())
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        if (step + 1) % 500 == 0 or step == steps - 1:
            print(step + 1, round(loss.item(), 3), 'val ppl', round(evaluate(m, D['valid'], T, CFG['sens_windows'])['ppl'], 2),
                  f'{time.time()-t0:.0f}s')
    return m

if os.path.exists('runs/fp_base.pt'):
    ck = torch.load('runs/fp_base.pt')
else:
    fp = pretrain(CFG['pretrain_steps'], CFG['pretrain_bs'], CFG['T'], CFG['pretrain_lr'])
    ck = {'cfg': fp.cfg, 'sd': {k: v.cpu() for k, v in fp.state_dict().items()}}
    torch.save(ck, 'runs/fp_base.pt')

def fresh():
    m = GPT(**ck['cfg']); m.load_state_dict(ck['sd']); return m.to(DEVICE).eval()

def quant_embedding(m, bits=8, g=64):
    w = m.wte.weight.data
    a = mse_clip_search(w, bits, g); s, z = minmax_params(w, bits, g, a)
    m.wte.weight.data = fake_quant(w, s, z, bits, g, ste=False)

TW = CFG['test_windows']
FP_TEST = evaluate(fresh(), D['test'], CFG['T'], TW); print('FP32 test', FP_TEST)

## 5. Which layers break first?

For every linear layer $\ell$ and bit-width $b\in\{2,\dots,6\}$ we quantise **only** that layer
(group-wise, MSE-clipped RTN) and record $\Delta\mathcal{L}_\ell(b)$, the increase in validation NLL.
We also compute cheap proxies (weight MSE, Hessian-weighted output error $\mathrm{tr}(\Delta W H \Delta W^\top)$,
Fisher-weighted error, kurtosis) and test how well they rank layers, and whether single-layer costs add up.

In [ ]:
NW = CFG['sens_windows']
EV = lambda m: evaluate(m, D['valid'], CFG['T'], max_windows=NW)['nll']
base = fresh(); nll0 = EV(base)
names = [m.name for m in base.qlinears()]; sizes = {m.name: m.weight.numel() for m in base.qlinears()}
g = torch.Generator().manual_seed(123)
calib = [get_batch(D['train'], 8, 256, g, DEVICE)[0] for _ in range(8)]
H = collect_inputs(base, calib)
fisher = empirical_fisher(base, D['train'], nb=16)
BITS = [2, 3, 4, 5, 6]
single, proxies = {}, {}
model = fresh(); mods = {m.name: m for m in model.qlinears()}
for n in names:
    m = mods[n]; W0 = m.weight.data.clone(); single[n] = {}
    proxies[n] = dict(kurtosis=kurtosis(W0.cpu()), max_over_std=(W0.abs().max() / W0.std()).item())
    for b in BITS:
        m.configure(b, 'frozen', CFG['group'], 'mse'); dW = m.qweight().detach() - W0
        single[n][b] = EV(model) - nll0
        proxies[n][f'w_mse_{b}'] = dW.pow(2).sum().item()
        proxies[n][f'hess_err_{b}'] = ((dW @ H[n].to(dW.device)) * dW).sum().item()
        proxies[n][f'fisher_err_{b}'] = 0.5 * (fisher[n] * dW.pow(2)).sum().item()
        m.qmode = None
print('done single-layer sweep')
S = pd.DataFrame(single).T; S.index.name = 'layer'
S.style.background_gradient(axis=None)

In [ ]:
types = ['q', 'k', 'v', 'o', 'fc1', 'fc2']; L = CFG['L']
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for i, b in enumerate([2, 3]):
    M = np.array([[single[f'{l}.{t}'][b] for t in types] for l in range(L)])
    im = ax[i].imshow(M, cmap='Blues'); ax[i].set_xticks(range(6), types); ax[i].set_yticks(range(L))
    ax[i].set_ylabel('block'); ax[i].set_title(f'single-layer $\\Delta$NLL at {b} bits'); plt.colorbar(im, ax=ax[i])
plt.tight_layout(); plt.show()

corr = {}
for b in [2, 3, 4]:
    y = [single[n][b] for n in names]
    for p in ['w_mse', 'hess_err', 'fisher_err']:
        corr[(p, b)] = spearmanr([proxies[n][f'{p}_{b}'] for n in names], y)[0]
    for p in ['kurtosis', 'max_over_std']:
        corr[(p, b)] = spearmanr([proxies[n][p] for n in names], y)[0]
pd.Series(corr).unstack()

In [ ]:
# groups: all layers of one type / all layers of one block; embedding; additivity
by_type, by_depth, emb = {}, {}, {}
for b in [2, 3]:
    for t in types:
        m = fresh(); quantize_model(m, {f'{l}.{t}': b for l in range(L)}, 'frozen'); by_type[(t, b)] = EV(m) - nll0
    for l in range(L):
        m = fresh(); quantize_model(m, {f'{l}.{t}': b for t in types}, 'frozen'); by_depth[(l, b)] = EV(m) - nll0
for b in [2, 3, 4, 6, 8]:
    m = fresh(); quant_embedding(m, b); emb[b] = EV(m) - nll0
print('by type', pd.Series(by_type).unstack()); print('by depth', pd.Series(by_depth).unstack()); print('embedding', emb)

rng = np.random.default_rng(0); add = []
for trial in range(5 if FAST else 30):
    choice = {n: int(rng.choice([2, 3, 4, 16], p=[.3, .3, .3, .1])) for n in names}
    pred = sum(single[n][b] for n, b in choice.items() if b < 16)
    m = fresh(); quantize_model(m, choice, 'frozen'); add.append((pred, EV(m) - nll0))
add = np.array(add)
plt.figure(figsize=(3.6, 3.4)); plt.scatter(add[:, 0], add[:, 1], s=14); lim = add.max() * 1.05
plt.plot([0, lim], [0, lim], 'k--', lw=1); plt.xlabel('sum of single-layer $\\Delta$NLL'); plt.ylabel('measured $\\Delta$NLL')
plt.title(f'additivity (Spearman {spearmanr(add[:,0], add[:,1])[0]:.2f})'); plt.show()

## 6. Post-training quantisation baselines

In [ ]:
def test(m, emb8=True):
    if emb8: quant_embedding(m, 8)
    return evaluate(m, D['test'], CFG['T'], TW)
calib16 = [get_batch(D['train'], 16, 256, torch.Generator().manual_seed(123), DEVICE)[0] for _ in range(8)]
PTQ = {}
for b in [4, 3, 2]:
    for gs in ([64] if FAST or CFG.get('smoke') else [32, 64, 128]):
        m = fresh(); quantize_model(m, b, 'frozen', gs, clip='none'); PTQ[('RTN', b, gs)] = test(m)['ppl']
        m = fresh(); quantize_model(m, b, 'frozen', gs, clip='mse'); PTQ[('RTN+clip', b, gs)] = test(m)['ppl']
        m = fresh(); gptq_quantize(m, calib16, b, gs); PTQ[('GPTQ', b, gs)] = test(m)['ppl']
pd.Series(PTQ).unstack([1, 2]).round(2)

## 7. Bit allocation by integer linear programming

$\min_{x}\sum_{\ell,b} c_\ell(b)\,x_{\ell b}$ s.t. $\sum_b x_{\ell b}=1$, $\sum_{\ell,b} n_\ell\, b\, x_{\ell b}\le \bar b \sum_\ell n_\ell$,
$x\in\{0,1\}$, with $b\in\{2,3,4\}$. SAMQ uses the *measured* single-layer cost $c_\ell(b)=\Delta\mathcal L_\ell(b)$.

In [ ]:
CH = [2, 3, 4]
def ilp(costfn, budget):
    return allocate_bits({n: {b: costfn(n, b) for b in CH} for n in names}, sizes, CH, budget)
def random_alloc(budget, seed):
    r = np.random.default_rng(seed)
    return allocate_bits({n: {b: float(r.random()) * (4 - b) for b in CH} for n in names}, sizes, CH, budget)
def heuristic_depth(budget):
    order = sorted(range(L), key=lambda l: min(l, L - 1 - l)); alloc = {n: 2 for n in names}; tot = sum(sizes.values())
    for b in [3, 4]:
        for l in order:
            for t in types:
                n = f'{l}.{t}'; trial = dict(alloc); trial[n] = b
                if sum(sizes[k] * v for k, v in trial.items()) / tot <= budget + 1e-9 and alloc[n] < b: alloc = trial
    return alloc
POL = {'ILP measured (SAMQ)': lambda B: ilp(lambda n, b: single[n][b], B),
       'ILP Fisher':          lambda B: ilp(lambda n, b: proxies[n][f'fisher_err_{b}'], B),
       'ILP Hessian':         lambda B: ilp(lambda n, b: proxies[n][f'hess_err_{b}'], B),
       'ILP weight-MSE':      lambda B: ilp(lambda n, b: proxies[n][f'w_mse_{b}'], B),
       'depth heuristic':     heuristic_depth,
       'random':              lambda B: random_alloc(B, 0)}
ALLOC, AP = {}, {}
for B in [2.25, 2.5, 3.0]:
    for pn, fn in POL.items():
        a = fn(B); ALLOC[(pn, B)] = a
        m = fresh(); gptq_quantize(m, calib16, a, 64); AP[(pn, B)] = test(m)['ppl']
pd.Series(AP).unstack().round(2)

In [ ]:
a = ALLOC[('ILP measured (SAMQ)', 2.5)]
M = np.array([[a[f'{l}.{t}'] for t in types] for l in range(L)])
plt.figure(figsize=(4, 3)); plt.imshow(M, cmap='Blues', vmin=1.5, vmax=4)
for l in range(L):
    for j in range(6): plt.text(j, l, M[l, j], ha='center', va='center')
plt.xticks(range(6), types); plt.ylabel('block'); plt.title('SAMQ bit allocation @ 2.5 bits'); plt.show()

## 8. Quantization-aware fine-tuning

* **STE** — min-max scales (MSE clip ratio fixed at init) recomputed every step, straight-through rounding, CE loss
* **LSQ** — learnable per-group log-step and zero-point, MSE-clipped initialisation, CE loss
* **LSQ+KD** — LSQ trained on the KL divergence to the FP teacher's logits
* **SAMQ** — LSQ+KD with the per-layer 2/3/4-bit allocation from the ILP over *measured* sensitivities (Sec. 7).
  At uniform bit-widths SAMQ coincides with LSQ+KD.
* **Candidate components** (rejected by the validation pilots in the paper, kept here as options): hidden-state
  distillation (`hid`), progressive 4→3→2 bit annealing (`prog`), GPTQ warm start (`init='gptq'`).
* **FP fine-tuned** — the unquantised model fine-tuned for the same number of steps (control for extra training).

Dropout is disabled in the student during fine-tuning (with p=0.1 it alone causes ~0.08 nats of teacher–student KL).

In [ ]:
RES = {}
def codes_of(m):
    with torch.no_grad():
        qmax = 2 ** m.bits - 1
        if m.qmode == 'lsq': s = m.log_s.exp(); z = m.zp.round().clamp(0, qmax)
        else: s, z = minmax_params(m.weight, m.bits, m.g, m.alpha)
        wg = m.weight.reshape(m.weight.shape[0], -1, m.g)
        return torch.clamp(torch.round(wg / s.unsqueeze(-1)) + z.unsqueeze(-1), 0, qmax).to(torch.int8)

def run_qat(name, bmap, method='lsq', kd=None, hid=0.0, prog=False, init='mse', seed=0, steps=None, g=None):
    steps = steps or CFG['qat_steps']; g = CFG['group'] if g is None else g
    if isinstance(bmap, int): bmap = {n: bmap for n in names}
    st = fresh()
    for mod in st.modules():                       # no dropout during fine-tuning
        if isinstance(mod, nn.Dropout): mod.p = 0.0
    clip = init
    if init == 'gptq':                             # warm start from the GPTQ solution
        gptq_quantize(st, calib16, bmap, g); clip = 'mse'
    quantize_model(st, {n: (max(b, 4) if prog else b) for n, b in bmap.items()},
                   {'ste': 'minmax', 'lsq': 'lsq'}[method], g, clip=clip)
    teacher = fresh() if kd else None
    osc = {n: 0.0 for n in names}; prev = {}; nobs = [0]
    def cb(step):
        if step >= int(0.75 * steps) and step % 5 == 0:
            for m in st.qlinears():
                if m.qmode is None: continue
                c = codes_of(m)
                if m.name in prev and prev[m.name].shape == c.shape: osc[m.name] += (c != prev[m.name]).float().mean().item()
                prev[m.name] = c
            nobs[0] += 1
    t0 = time.time()
    hist = qat_train(st, D['train'], D['valid'], steps=steps, bs=CFG['qat_bs'], T=CFG['T'], lr=CFG['qat_lr'],
                     qlr=CFG['qat_qlr'], teacher=teacher, kd=kd, hid_w=hid, seed=seed,
                     anneal=dict(final=bmap, start_bits=4, frac=0.5) if prog else None,
                     log_every=max(50, steps // 3), eval_windows=min(64, CFG['sens_windows']), callback=cb)
    r = dict(bits_report(st, g), hist=hist, time=time.time() - t0,
             osc={n: v / max(1, nobs[0] - 1) for n, v in osc.items()})
    quant_embedding(st, 8)
    r['test'] = evaluate(st, D['test'], CFG['T'], TW); r['valid'] = evaluate(st, D['valid'], CFG['T'], TW)
    fpm = fresh()
    with torch.no_grad():
        x, _ = get_batch(D['valid'], 16, CFG['T'], torch.Generator().manual_seed(7), DEVICE)
        _, hs = st(x, return_hidden=True); _, ht = fpm(x, return_hidden=True)
        r['hid_rel_err'] = [((a_ - b_).pow(2).mean() / b_.pow(2).mean()).item() for a_, b_ in zip(hs, ht)]
    RES[name] = r
    print(f'{name:26s} avg bits {r["avg_bits"]:.2f}  test PPL {r["test"]["ppl"]:.2f}  ({r["time"]:.0f}s)')
    return st

K = dict(kd='logit')
MP = lambda B: ilp(lambda n, b: single[n][b], B)
EXPERIMENTS = [
    ('FP fine-tuned', 16, {}),
    ('STE W2', 2, dict(method='ste')), ('LSQ W2', 2, {}), ('LSQ+KD W2 (=SAMQ W2)', 2, K),
    ('SAMQ MP2.25', MP(2.25), K), ('SAMQ MP2.5', MP(2.5), K), ('SAMQ MP3.0', MP(3.0), K),
    ('LSQ+KD W3 (=SAMQ W3)', 3, K),
    ('alloc Fisher-ILP 2.5', ilp(lambda n, b: proxies[n][f'fisher_err_{b}'], 2.5), K),
    ('alloc random 2.5', random_alloc(2.5, 0), K),
    ('cand: + GPTQ warm start W2', 2, dict(kd='logit', init='gptq')),
    ('cand: + hidden KD W2', 2, dict(kd='logit', hid=1.0)),
    ('cand: + progressive W2', 2, dict(kd='logit', prog=True)),
    ('LSQ+KD W2 per-channel', 2, dict(kd='logit', g=0)),
    ('LSQ+KD W4', 4, K),
]
if FAST: EXPERIMENTS = EXPERIMENTS[:6]
for name, bm, kw in EXPERIMENTS:
    if name not in RES: run_qat(name, bm, **kw)

## 9. Results

In [ ]:
rows = [dict(method='FP32', avg_bits=32, test_ppl=FP_TEST['ppl'], acc=FP_TEST['acc'])]
for k, r in RES.items():
    rows.append(dict(method=k, avg_bits=r['avg_bits'], eff_bits=r['eff_bits'], test_ppl=r['test']['ppl'], acc=r['test']['acc']))
T = pd.DataFrame(rows).set_index('method').round(3); T

In [ ]:
plt.figure(figsize=(5, 3.6))
fam = {'STE W': '#2a78d6', 'LSQ W': '#eb6834', 'LSQ+KD W': '#1baf7a', 'SAMQ MP': '#4a3aa7'}
for f, c in fam.items():
    pts = sorted([(r['avg_bits'], r['test']['ppl']) for k, r in RES.items() if k.startswith(f) and 'per-channel' not in k])
    if pts: plt.plot(*zip(*pts), 'o-', color=c, label=f.strip(), lw=2, ms=6)
for (meth, b, gs), v in PTQ.items():
    if meth == 'GPTQ' and gs == 64: plt.scatter(b, v, marker='s', color='#52514e', label='GPTQ' if b == 2 else None)
plt.axhline(FP_TEST['ppl'], color='k', ls=':', lw=1, label='FP32')
plt.yscale('log'); plt.xlabel('average weight bits'); plt.ylabel('test perplexity'); plt.legend(); plt.show()

plt.figure(figsize=(5, 3.4))
for k in ['STE W2', 'LSQ W2', 'LSQ+KD W2 (=SAMQ W2)', 'SAMQ MP2.5']:
    if k in RES: plt.plot(range(L), RES[k]['hid_rel_err'], 'o-', label=k)
plt.yscale('log'); plt.xlabel('block'); plt.ylabel('relative hidden-state error'); plt.legend(); plt.show()

## 10. Real bit-packed export and edge projections

In [ ]:
def packed_size(model_q, bmap, g=64):
    '''Exact bytes of a bit-packed checkpoint: b-bit codes + fp16 scales + b-bit zeros per group,
    8-bit embedding (tied head) with fp16 per-group scales, fp16 norms/biases/positions.'''
    tot = 0
    for m in model_q.qlinears():
        b = bmap[m.name]; W = m.weight.data.cpu()
        a = mse_clip_search(W, b, g); s, z = minmax_params(W, b, g, a)
        e = export_linear(W, s, z, b, g); tot += nbytes(e)
        assert torch.allclose(dequant_linear(e), fake_quant(W, s.half().float(), z, b, g, ste=False), atol=1e-6)
    W = model_q.wte.weight.data.cpu(); s, z = minmax_params(W, 8, g)
    tot += nbytes(export_linear(W, s, z, 8, g))
    other = sum(p.numel() for n, p in model_q.named_parameters() if p.dim() < 2 or 'wpe' in n)
    return tot + 2 * other
fp_bytes = 4 * nparams(fresh())
for B in [2, 3, 4]:
    print(f'W{B}: {packed_size(fresh(), {n: B for n in names})/1e6:.2f} MB vs FP32 {fp_bytes/1e6:.2f} MB')
print(f"SAMQ MP2.5: {packed_size(fresh(), MP(2.5))/1e6:.2f} MB")

# bandwidth-bound decode projection (analytical): tokens/s ~= memory bandwidth / bytes read per token
for P in [1e9, 3e9, 7e9]:
    for bits in [16, 4, 3, 2.5, 2]:
        eff = bits + (16 + bits) / 64 if bits < 16 else 16
        gb = P * eff / 8 / 1e9
        print(f'{P/1e9:.0f}B @ {bits:>4} bit: {gb:5.2f} GB, ~{60/gb:5.1f} tok/s at 60 GB/s')

## 11. *(Optional, GPU)* The same pipeline on a pretrained Hugging Face LLM

Wraps every `nn.Linear` in the decoder layers as a `QLinear`, measures single-layer sensitivity at 2/3/4 bits on
WikiText-2, solves the ILP and runs SAMQ fine-tuning with the FP model as teacher.
Defaults to `HuggingFaceTB/SmolLM2-135M` (fits a free T4); `Qwen/Qwen2.5-0.5B` also works.

In [ ]:
"""hf_ext.py -- apply the same sensitivity / ILP / SAMQ pipeline to a Hugging Face causal LM."""
import torch, torch.nn as nn, torch.nn.functional as F

def wrap_hf_linears(hf_model):
    """Replace every nn.Linear inside the decoder layers with a QLinear (weights shared).
    Embeddings and lm_head are left untouched (kept at 8 bit at export, as in the paper)."""
    layers = hf_model.model.layers
    for li, layer in enumerate(layers):
        for pname, mod in list(layer.named_modules()):
            if type(mod) is nn.Linear:
                q = QLinear(mod.in_features, mod.out_features, bias=mod.bias is not None)
                q = q.to(device=mod.weight.device, dtype=mod.weight.dtype)
                q.weight = mod.weight
                if mod.bias is not None: q.bias = mod.bias
                q.name = f'{li}.{pname}'
                parent = layer
                *path, last = pname.split('.')
                for p in path: parent = getattr(parent, p)
                setattr(parent, last, q)
    return hf_model

class HFLM(nn.Module):
    """Adapter exposing the interface used by qlib (logits, optional hidden states, qlinears)."""
    def __init__(self, hf_model):
        super().__init__(); self.m = hf_model
    def forward(self, idx, return_hidden=False):
        out = self.m(input_ids=idx, output_hidden_states=return_hidden, use_cache=False)
        if return_hidden:
            return out.logits.float(), [h.float() for h in out.hidden_states[1:]]
        return out.logits.float()
    def qlinears(self):
        return [m for m in self.modules() if isinstance(m, QLinear)]

@torch.no_grad()
def gptq_quantize_hf(lm, calib, bits_map, g=64):
    """Layer-sequential GPTQ for an arbitrary HF decoder: for decoder layer l, collect input
    Hessians of all its linears with layers < l already quantised, then quantise layer l."""
    lm.eval()
    by_layer = {}
    for m in lm.qlinears(): by_layer.setdefault(int(m.name.split('.')[0]), []).append(m)
    for li in sorted(by_layer):
        mods = by_layer[li]
        Hs, cnt, hooks = {}, {}, []
        for mod in mods:
            def hook(mod, inp, out):
                a = inp[0].reshape(-1, inp[0].shape[-1]).float()
                Hs[mod.name] = Hs.get(mod.name, 0) + 2 * a.t() @ a
                cnt[mod.name] = cnt.get(mod.name, 0) + a.shape[0]
            hooks.append(mod.register_forward_hook(hook))
        for x in calib: lm(x)
        for h in hooks: h.remove()
        for mod in mods:
            b = bits_map if isinstance(bits_map, int) else bits_map.get(mod.name, 16)
            if b >= 16: continue
            Wq = gptq_layer(mod.weight.data.float(), Hs[mod.name] / cnt[mod.name], b, g)
            mod.weight.data = Wq.to(mod.weight.dtype)
    return lm

In [ ]:
RUN_HF = False   # set True on a GPU runtime
if RUN_HF:
    !pip -q install transformers datasets accelerate
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from datasets import load_dataset
    MID = 'HuggingFaceTB/SmolLM2-135M'
    tok = AutoTokenizer.from_pretrained(MID)
    ds = load_dataset('wikitext', 'wikitext-2-raw-v1')
    enc = lambda split: torch.tensor(tok('\n\n'.join(ds[split]['text']))['input_ids'], dtype=torch.int32)
    HD = {s: enc(s) for s in ['train', 'validation', 'test']}
    def load():
        return HFLM(wrap_hf_linears(AutoModelForCausalLM.from_pretrained(MID, torch_dtype=torch.float32))).to(DEVICE).eval()
    teacher = load(); SEQ = 512
    hnames = [m.name for m in teacher.qlinears()]; hsizes = {m.name: m.weight.numel() for m in teacher.qlinears()}
    print('FP PPL', evaluate(teacher, HD['test'], SEQ, bs=8)['ppl'])
    # sensitivity (2/3/4 bits, single layer, 32 validation windows)
    probe = load(); pm = {m.name: m for m in probe.qlinears()}
    ev = lambda m: evaluate(m, HD['validation'], SEQ, max_windows=32, bs=8)['nll']; n0 = ev(probe)
    hsens = {}
    for n in hnames:
        hsens[n] = {}
        for b in [2, 3, 4]:
            pm[n].configure(b, 'frozen', 64); hsens[n][b] = ev(probe) - n0; pm[n].qmode = None
    hb = allocate_bits(hsens, hsizes, [2, 3, 4], 2.5)
    calib = [get_batch(HD['train'], 4, SEQ, torch.Generator().manual_seed(0), DEVICE)[0] for _ in range(32)]
    g = load(); gptq_quantize_hf(g, calib, hb); print('GPTQ-MP2.5 PPL', evaluate(g, HD['test'], SEQ, bs=8)['ppl'])
    st = load(); quantize_model(st, {n: 4 for n in hnames}, 'lsq', 64)
    qat_train(st, HD['train'], HD['validation'], steps=1000, bs=4, T=SEQ, lr=1e-5, qlr=1e-3, teacher=teacher,
              kd='logit', hid_w=CFG['hid_w'], anneal=dict(final=hb, start_bits=4, frac=0.5), log_every=200)
    print('SAMQ-MP2.5 PPL', evaluate(st, HD['test'], SEQ, bs=8)['ppl'])